# 6장. 명령 한 번으로 끝내기

2~5장의 함수를 `weekly_report.py` 한 파일에 모아 두었다. 여기서는 그 파일을 불러 쓰고,
일부러 틀린 상황을 만들어 어떤 안내가 나오는지 확인한다.

In [1]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")  # 한 칸 위 폴더의 weekly_report.py를 불러오기 위해
import weekly_report as wr

## 1. 함수 하나로 보고서 만들기

In [2]:
path, notes = wr.make_report("2026-11-02")

print("만든 파일:", path.name)
for note in notes:
    print(" -", note)

만든 파일: 주간판매보고서_2026-11-02.xlsx
 - 중복 전표 제거: 광주점 3건
 - 수원점 합계 검산: 파일 5,975,100원 / 계산 5,975,100원 → 일치
 - 확인이 필요한 줄 1건 ('확인필요' 시트)
 - 판매 253건, 합계 29,919,700원


## 2. 터미널 명령으로 실행하기
터미널에서 `python run_report.py 2026-11-02`라고 치는 것과 같은 일을 여기서 대신 해 본다.

In [3]:
result = subprocess.run(
    [sys.executable, "-X", "utf8", "run_report.py", "2026-11-02"],
    cwd="..", capture_output=True, text=True, encoding="utf-8",
)
print(result.stdout)
print("종료 코드:", result.returncode)

2026-11-02 주 보고서를 만들었습니다: output/주간판매보고서_2026-11-02.xlsx
 - 중복 전표 제거: 광주점 3건
 - 수원점 합계 검산: 파일 5,975,100원 / 계산 5,975,100원 → 일치
 - 확인이 필요한 줄 1건 ('확인필요' 시트)
 - 판매 253건, 합계 29,919,700원

종료 코드: 0


날짜를 안 주면 오늘 기준 지난주 월요일을 스스로 계산한다. 계산이 맞는지 날짜 몇 개로 확인한다.

In [4]:
import datetime as dt

import run_report

for day in ["2026-11-09", "2026-11-11", "2026-11-15", "2026-11-16"]:
    today = dt.date.fromisoformat(day)
    print(day, "에 실행하면 →", run_report.last_monday(today), "주 보고서")

2026-11-09 에 실행하면 → 2026-11-02 주 보고서
2026-11-11 에 실행하면 → 2026-11-02 주 보고서
2026-11-15 에 실행하면 → 2026-11-02 주 보고서
2026-11-16 에 실행하면 → 2026-11-09 주 보고서


## 3. 일부러 틀려 보기
자동화는 잘 될 때보다 안 될 때가 중요하다. 연습용 임시 폴더에 데이터를 복사해 놓고 세 가지를 망가뜨려 본다.

In [5]:
sandbox = Path(tempfile.mkdtemp())
shutil.copytree(wr.DATA / "raw", sandbox / "raw")
shutil.copytree(wr.DATA / "master", sandbox / "master")
out = sandbox / "output"


def try_report(week):
    try:
        path, notes = wr.make_report(week, data_dir=sandbox, output_dir=out)
        print("만듦:", path.name)
        for note in notes:
            print(" -", note)
    except (FileNotFoundError, ValueError) as error:
        print("멈춤:", error)

### 가. 아직 파일을 안 넣은 주

In [6]:
try_report("2026-11-09")

멈춤: raw/2026-11-09 폴더가 없습니다. 지점 파일을 이 폴더에 넣었는지 확인하세요.


### 나. 지점 하나가 파일을 안 보낸 주
멈추지 않고 만들되, 빠진 지점을 메모에 남긴다.

In [7]:
(sandbox / "raw" / "2026-11-02" / "대전점_주간판매.xlsx").unlink()
try_report("2026-11-02")

만듦: 주간판매보고서_2026-11-02.xlsx
 - 중복 전표 제거: 광주점 3건
 - 수원점 합계 검산: 파일 5,975,100원 / 계산 5,975,100원 → 일치
 - 확인이 필요한 줄 1건 ('확인필요' 시트)
 - 파일이 오지 않은 지점: 대전점
 - 판매 213건, 합계 25,859,000원


### 다. 열 이름이 바뀐 파일
강남점이 '금액' 열 이름을 '매출'로 바꿔 보냈다고 하자.

In [8]:
broken = sandbox / "raw" / "2026-11-02" / "강남점_주간판매.xlsx"
table = pd.read_excel(broken).rename(columns={"금액": "매출"})
table.to_excel(broken, index=False)
try_report("2026-11-02")

멈춤: 강남점_주간판매.xlsx: 필요한 열이 없습니다 → ['금액']


### 라. 월요일이 아닌 날짜

In [9]:
try_report("2026-11-03")

멈춤: 2026-11-03은 월요일이 아닙니다. 보고할 주의 월요일 날짜를 넣으세요.


In [10]:
shutil.rmtree(sandbox)
print("연습용 임시 폴더를 지웠습니다.")

연습용 임시 폴더를 지웠습니다.


## 실습. 실행 기록 확인하기
`run_report.py`는 실행할 때마다 `output/run_log.txt`에 한 줄씩 남긴다. 예약 실행을 걸어 두면 이 파일이 유일한 흔적이다.

In [11]:
log_lines = (wr.OUTPUT / "run_log.txt").read_text(encoding="utf-8").splitlines()
print(log_lines[-1][17:])  # 앞 17글자는 실행한 날짜와 시각

[성공] 2026-11-02 중복 전표 제거: 광주점 3건 / 수원점 합계 검산: 파일 5,975,100원 / 계산 5,975,100원 → 일치 / 확인이 필요한 줄 1건 ('확인필요' 시트) / 판매 253건, 합계 29,919,700원
